# Clasificación Binaria de reingreso de un paciente en 30 dias en un Hospital

**Objetivo:** La idea de este proyecto es poder predecir el reingreso en 30 días de un paciente a una hospitalización:

### Base de datos
- `Hospital Management Data`: https://www.kaggle.com/datasets/deeplumiere/hospital-management-data/data

- `Licencia`: MIT - https://www.mit.edu/~amini/LICENSE.md 



### Archivos de Origen
- `visit_encounters.csv`: Información de la hospitalización.
- `clinical_details.csv`: Información detallada de la hospitalización.
- `hospital_billing.csv`: Facturación de la hospitalización.
- `patients.csv`: Información del paciente.
- `pharmacy_prescriptions.csv`: Medicametos recetados durante la hospitalización.



## Librerias y Configuración

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## Análisis Exploratorio de cada Dataset

Para entender el contexto de cada uno de los datasets y asi poder definir el dataset final entonces haremos un análisis individual para ver su cantidad, calidad y distribución de los datos.

In [24]:
# Cargar de los datasets
patients   = pd.read_csv("../data/patients.csv")
encounters = pd.read_csv("../data/visits_encounters.csv")
clinical   = pd.read_csv("../data/clinical_details.csv")
billing    = pd.read_csv("../data/hospital_billing.csv")
pharmacy   = pd.read_csv("../data/pharmacy_prescriptions.csv")

datasets = {
    "patients": patients,
    "visits_encounters": encounters,
    "clinical_details": clinical,
    "hospital_billing": billing,
    "pharmacy_prescriptions": pharmacy
}

resumen = []
for name, df in datasets.items():
    resumen.append({
        "Dataset": name,
        "Filas" : df.shape[0],
        "Columnas" : df.shape[1],
        "Faltantes" : round(df.isnull().mean().mean()*100,2)
    })
resumen_df = pd.DataFrame(resumen)
resumen_df

,Dataset,Filas,Columnas,Faltantes
0,patients,5000,21,1.46
1,visits_encounters,13214,18,4.81
2,clinical_details,13214,22,3.53
3,hospital_billing,13214,18,5.28
4,pharmacy_prescriptions,26715,16,0.32


## Patients

In [25]:
patients.head()

,patient_id,first_name,last_name,gender,date_of_birth,age,blood_group,marital_status,race_ethnicity,residential_zone,socioeconomic_tier,primary_insurance_provider,insurance_policy_id,contact_phone,emergency_contact_phone,street_address,zip_code,baseline_bmi,smoker_status,baseline_charlson_comorbidity_index,registration_date
0,PAT-10001,Mark,Johnson,Male,1945-11-05,77,O+,Married,Hispanic,Suburban,High,Medicare,INS-887201,533-521-8196x001,790.683.8637x94026,51161 Abbott Rapids,35172,35.70,Former,1,2021-10-02
1,PAT-10002,Renee,Blair,Female,1982-12-16,40,B-,Married,Caucasian,Suburban,Middle,Self-Pay,NaN,+1-516-318-4959,NaN,3419 Amanda Gardens Apt. 764,84387,30.00,Never,3,2021-08-24
2,PAT-10003,Melissa,Robinson,Female,2004-10-09,18,O+,Single,African American,Suburban,Middle,BlueCross,INS-560337,903-405-6413x95376,423-988-4969x653,12269 Paul Ranch,61849,21.10,Never,1,2021-05-30
3,PAT-10004,Kendra,Maddox,Female,1942-02-10,80,A+,Divorced,Caucasian,Urban,Low,Medicare,INS-123247,9182451462,NaN,15430 Natalie Common Apt. 718,16984,32.60,Former,4,2021-01-31
4,PAT-10005,Betty,Alvarez,Female,2004-03-10,18,A+,Single,Hispanic,Urban,Middle,BlueCross,INS-384806,+1-624-889-6383,+1-557-287-1331x50983,031 Michele Cliffs,67892,33.30,Current,1,2021-02-18


In [26]:
patients.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 21 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   patient_id                           5000 non-null   str    
 1   first_name                           5000 non-null   str    
 2   last_name                            5000 non-null   str    
 3   gender                               5000 non-null   str    
 4   date_of_birth                        5000 non-null   str    
 5   age                                  5000 non-null   int64  
 6   blood_group                          5000 non-null   str    
 7   marital_status                       5000 non-null   str    
 8   race_ethnicity                       5000 non-null   str    
 9   residential_zone                     5000 non-null   str    
 10  socioeconomic_tier                   5000 non-null   str    
 11  primary_insurance_provider           5000

In [27]:
patients.describe(include='all')

,patient_id,first_name,last_name,gender,date_of_birth,age,blood_group,marital_status,race_ethnicity,residential_zone,socioeconomic_tier,primary_insurance_provider,insurance_policy_id,contact_phone,emergency_contact_phone,street_address,zip_code,baseline_bmi,smoker_status,baseline_charlson_comorbidity_index,registration_date
count,5000,5000,5000,5000,5000,5000.00,5000,5000,5000,5000,5000,5000,4598,4714,4709,5000,5000.00,4703.00,4746,5000.00,5000
unique,5000,589,915,2,4563,NaN,8,4,5,3,3,6,4587,4714,4709,5000,NaN,NaN,3,NaN,550
top,PAT-10001,Michael,Smith,Female,1971-12-20,NaN,O+,Married,Caucasian,Urban,Middle,BlueCross,INS-728138,533-521-8196x001,790.683.8637x94026,51161 Abbott Rapids,NaN,NaN,Never,NaN,2021-09-16
freq,1,117,99,2514,3,NaN,1899,2296,2966,2722,2443,1318,2,1,1,1,NaN,NaN,2994,NaN,20
mean,NaN,NaN,NaN,NaN,NaN,44.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49882.47,28.39,NaN,1.97,NaN
std,NaN,NaN,NaN,NaN,NaN,23.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28891.06,4.38,NaN,1.69,NaN
min,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,507.00,15.00,NaN,0.00,NaN
25%,NaN,NaN,NaN,NaN,NaN,27.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24534.50,25.40,NaN,1.00,NaN
50%,NaN,NaN,NaN,NaN,NaN,47.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49866.00,28.50,NaN,2.00,NaN
75%,NaN,NaN,NaN,NaN,NaN,65.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74702.50,31.40,NaN,3.00,NaN


## Visit Encounters

In [28]:
encounters.head()

,encounter_id,patient_id,encounter_type,admission_datetime,discharge_datetime,length_of_stay_days,admission_urgency,department,attending_physician_id,icu_admitted,icu_days,discharge_disposition,news2_acuity_score,revisit_count,previous_encounter_id,days_since_last_discharge,readmission_30d,in_hospital_mortality
0,ENC-100001,PAT-10001,Emergency,2023-03-30 00:00:00,2023-04-03 07:12:00,4.30,Emergency,General Medicine,DOC-106,0,0.00,Skilled Nursing Facility,1,1,NaN,NaN,0,0
1,ENC-100002,PAT-10001,Outpatient,2023-04-06 12:00:00,2023-04-06 16:19:12,0.18,Elective,General Medicine,DOC-151,0,0.00,Home,2,2,ENC-100001,3.20,0,0
2,ENC-100003,PAT-10001,Outpatient,2023-04-18 04:19:12,2023-04-18 13:55:12,0.40,Elective,Cardiology,DOC-111,0,0.00,Home,9,3,ENC-100002,11.50,1,0
3,ENC-100004,PAT-10002,Emergency,2023-03-28 00:00:00,2023-04-07 19:12:00,10.80,Emergency,Cardiology,DOC-115,0,0.00,Home,7,1,NaN,NaN,1,0
4,ENC-100005,PAT-10002,Inpatient,2023-04-09 12:00:00,2023-04-12 16:48:00,3.20,Urgent,General Medicine,DOC-134,0,0.00,Home,0,2,ENC-100004,1.70,1,0


In [29]:
encounters.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 13214 entries, 0 to 13213
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   encounter_id               13214 non-null  str    
 1   patient_id                 13214 non-null  str    
 2   encounter_type             13214 non-null  str    
 3   admission_datetime         13214 non-null  str    
 4   discharge_datetime         13214 non-null  str    
 5   length_of_stay_days        13214 non-null  float64
 6   admission_urgency          13214 non-null  str    
 7   department                 13214 non-null  str    
 8   attending_physician_id     12475 non-null  str    
 9   icu_admitted               13214 non-null  int64  
 10  icu_days                   13214 non-null  float64
 11  discharge_disposition      12509 non-null  str    
 12  news2_acuity_score         13214 non-null  int64  
 13  revisit_count              13214 non-null  int64  
 14  p

In [30]:
encounters.describe(include='all')

,encounter_id,patient_id,encounter_type,admission_datetime,discharge_datetime,length_of_stay_days,admission_urgency,department,attending_physician_id,icu_admitted,icu_days,discharge_disposition,news2_acuity_score,revisit_count,previous_encounter_id,days_since_last_discharge,readmission_30d,in_hospital_mortality
count,13214,13214,13214,13214,13214,13214.00,13214,13214,12475,13214.00,13214.00,12509,13214.00,13214.00,8214,8214.00,13214.00,13214.00
unique,13214,5000,3,7161,9867,NaN,3,8,60,NaN,NaN,6,NaN,NaN,8214,NaN,NaN,NaN
top,ENC-100001,PAT-10013,Outpatient,2023-03-05 00:00:00,2023-06-25 16:48:00,NaN,Elective,General Medicine,DOC-131,NaN,NaN,Home,NaN,NaN,ENC-100001,NaN,NaN,NaN
freq,1,6,5389,45,9,NaN,6423,4918,240,NaN,NaN,9388,NaN,NaN,1,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,5.76,NaN,NaN,NaN,0.04,0.21,NaN,5.50,2.28,NaN,106.58,0.33,0.10
std,NaN,NaN,NaN,NaN,NaN,7.06,NaN,NaN,NaN,0.20,1.22,NaN,3.89,1.37,NaN,103.51,0.47,0.30
min,NaN,NaN,NaN,NaN,NaN,0.15,NaN,NaN,NaN,0.00,0.00,NaN,0.00,1.00,NaN,1.00,0.00,0.00
25%,NaN,NaN,NaN,NaN,NaN,0.33,NaN,NaN,NaN,0.00,0.00,NaN,2.00,1.00,NaN,10.30,0.00,0.00
50%,NaN,NaN,NaN,NaN,NaN,3.60,NaN,NaN,NaN,0.00,0.00,NaN,5.00,2.00,NaN,70.35,0.00,0.00
75%,NaN,NaN,NaN,NaN,NaN,8.50,NaN,NaN,NaN,0.00,0.00,NaN,8.00,3.00,NaN,196.80,1.00,0.00


## Clinical Details

In [31]:
clinical.head()

,clinical_record_id,encounter_id,patient_id,primary_diagnosis_code,primary_diagnosis_name,disease_category,disease_severity,secondary_diagnosis_code,systolic_bp_mmhg,diastolic_bp_mmhg,heart_rate_bpm,respiratory_rate_bpm,temperature_celsius,spo2_percent,blood_glucose_mg_dl,serum_creatinine_mg_dl,hemoglobin_g_dl,white_blood_cell_count,primary_procedure_code,primary_procedure_name,procedure_complexity,hospital_acquired_infection
0,CLN-100001,ENC-100001,PAT-10001,N39.0,Urinary Tract Infection with Pyelonephritis,Renal,Mild,E78.5,119.70,78.90,67.30,16.70,36.60,99.60,116.20,1.17,NaN,4.50,99221,Inpatient Bladder Management & Culture Evaluation,NaN,0
1,CLN-100002,ENC-100002,PAT-10001,J44.1,COPD with Acute Exacerbation,Respiratory,Moderate,N18.3,118.50,86.20,74.40,18.00,37.20,97.90,98.00,0.71,14.40,6.80,94660,Continuous Positive Airway Pressure (CPAP) Man...,Minor,0
2,CLN-100003,ENC-100003,PAT-10001,I50.9,"Congestive Heart Failure, Unspecified",Cardiovascular,Severe,E78.5,129.00,87.20,98.70,15.20,36.70,98.80,118.50,2.89,15.10,25.90,99223,Comprehensive Inpatient Medical Management,NaN,0
3,CLN-100004,ENC-100004,PAT-10002,I21.9,Acute Myocardial Infarction,Cardiovascular,Severe,I10,150.80,102.00,94.00,19.10,37.00,98.50,92.80,1.46,13.90,22.30,92928,Percutaneous Coronary Angioplasty with Stent,Major,0
4,CLN-100005,ENC-100005,PAT-10002,J18.9,Community-Acquired Pneumonia,Respiratory,Mild,E66.0,120.90,81.40,59.30,14.60,36.70,98.60,66.50,0.83,NaN,7.20,99223,Inpatient Pulmonary Evaluation & Sputum Culture,NaN,1


In [32]:
clinical.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 13214 entries, 0 to 13213
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   clinical_record_id           13214 non-null  str    
 1   encounter_id                 13214 non-null  str    
 2   patient_id                   13214 non-null  str    
 3   primary_diagnosis_code       13214 non-null  str    
 4   primary_diagnosis_name       13214 non-null  str    
 5   disease_category             13214 non-null  str    
 6   disease_severity             13214 non-null  str    
 7   secondary_diagnosis_code     12423 non-null  str    
 8   systolic_bp_mmhg             13214 non-null  float64
 9   diastolic_bp_mmhg            13214 non-null  float64
 10  heart_rate_bpm               13214 non-null  float64
 11  respiratory_rate_bpm         13214 non-null  float64
 12  temperature_celsius          13214 non-null  float64
 13  spo2_percent               

In [33]:
clinical.describe(include='all')

,clinical_record_id,encounter_id,patient_id,primary_diagnosis_code,primary_diagnosis_name,disease_category,disease_severity,secondary_diagnosis_code,systolic_bp_mmhg,diastolic_bp_mmhg,heart_rate_bpm,respiratory_rate_bpm,temperature_celsius,spo2_percent,blood_glucose_mg_dl,serum_creatinine_mg_dl,hemoglobin_g_dl,white_blood_cell_count,primary_procedure_code,primary_procedure_name,procedure_complexity,hospital_acquired_infection
count,13214,13214,13214,13214,13214,13214,13214,12423,13214.00,13214.00,13214.00,13214.00,13214.00,12459.00,12391.00,12459.00,12540.00,12412.00,13214.00,13214,7560,13214.00
unique,13214,13214,5000,13,13,11,4,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13,4,NaN
top,CLN-100001,ENC-100001,PAT-10013,J18.9,Community-Acquired Pneumonia,Respiratory,Moderate,M54.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inpatient Pulmonary Evaluation & Sputum Culture,Moderate,NaN
freq,1,1,6,1695,1695,2498,5104,1603,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1695,3270,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122.21,80.82,80.56,17.50,36.94,96.85,112.94,1.59,13.08,11.12,72073.65,NaN,NaN,0.03
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.06,12.43,16.99,4.13,0.73,3.44,39.56,0.84,2.03,5.73,30525.20,NaN,NaN,0.16
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.00,45.00,55.00,12.00,36.00,77.90,65.00,0.50,6.50,3.20,27236.00,NaN,NaN,0.00
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,112.90,74.10,69.70,15.00,36.50,96.60,91.30,0.98,11.80,6.90,36556.00,NaN,NaN,0.00
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122.30,80.80,76.80,16.50,36.80,97.80,104.20,1.29,13.30,8.70,92928.00,NaN,NaN,0.00
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,131.60,87.60,85.80,18.40,37.10,98.80,119.00,2.01,14.50,15.50,99222.00,NaN,NaN,0.00


## Hospital Billing

In [34]:
billing.head()

,bill_id,encounter_id,patient_id,billing_date,room_and_board_charges,physician_consultation_charges,procedure_surgical_charges,laboratory_diagnostic_charges,pharmacy_charges,gross_total_amount,discount_contractual_adjustment,insurance_covered_amount,patient_deductible_copay,patient_paid_amount,outstanding_balance,payment_status,claim_status,claim_denial_reason
0,BIL-100001,ENC-100001,PAT-10001,2023-04-03,4085.00,1614.05,0.00,396.27,100.72,6196.04,1861.41,0.00,4334.63,3757.41,577.22,Partial Payment,Rejected,Timely Filing Limit Exceeded
1,BIL-100002,ENC-100002,PAT-10001,2023-04-07,250.00,358.02,1115.96,325.47,56.40,2105.85,717.26,1125.03,263.56,227.30,36.26,Partial Payment,Approved,NaN
2,BIL-100003,ENC-100003,PAT-10001,2023-04-18,250.00,314.99,0.00,3069.94,14.40,3649.33,1121.55,2382.37,145.41,124.82,20.59,Partial Payment,Approved,NaN
3,BIL-100004,ENC-100004,PAT-10002,2023-04-07,10260.00,4338.07,17609.75,2406.66,4.73,34619.21,6269.29,0.00,28349.92,18669.09,9680.83,Partial Payment,Self-Pay / Not Applicable,NaN
4,BIL-100005,ENC-100005,PAT-10002,2023-04-13,3040.00,937.05,0.00,439.05,89.55,4505.65,698.86,0.00,3806.79,2488.80,1317.99,Partial Payment,Self-Pay / Not Applicable,NaN


In [35]:
billing.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 13214 entries, 0 to 13213
Data columns (total 18 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   bill_id                          13214 non-null  str    
 1   encounter_id                     13214 non-null  str    
 2   patient_id                       13214 non-null  str    
 3   billing_date                     13214 non-null  str    
 4   room_and_board_charges           13214 non-null  float64
 5   physician_consultation_charges   13214 non-null  float64
 6   procedure_surgical_charges       13214 non-null  float64
 7   laboratory_diagnostic_charges    13214 non-null  float64
 8   pharmacy_charges                 13214 non-null  float64
 9   gross_total_amount               13214 non-null  float64
 10  discount_contractual_adjustment  12438 non-null  float64
 11  insurance_covered_amount         13214 non-null  float64
 12  patient_deductible_copay     

In [36]:
billing.describe(include='all')

,bill_id,encounter_id,patient_id,billing_date,room_and_board_charges,physician_consultation_charges,procedure_surgical_charges,laboratory_diagnostic_charges,pharmacy_charges,gross_total_amount,discount_contractual_adjustment,insurance_covered_amount,patient_deductible_copay,patient_paid_amount,outstanding_balance,payment_status,claim_status,claim_denial_reason
count,13214,13214,13214,13214,13214.00,13214.00,13214.00,13214.00,13214.00,13214.00,12438.00,13214.00,13214.00,13214.00,13214.00,13214,13214,1436
unique,13214,13214,5000,907,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,3,4
top,BIL-100001,ENC-100001,PAT-10013,2023-06-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Partial Payment,Approved,Prior Authorization Missing
freq,1,1,6,61,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12989,10686,385
mean,NaN,NaN,NaN,NaN,6145.09,2110.99,7870.33,1115.44,1387.89,18629.73,5724.81,8611.04,4299.18,2331.89,1967.29,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,8783.67,2421.57,11486.07,1009.12,2781.08,18097.32,5726.39,10422.94,8497.07,5232.60,4984.88,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,250.00,280.02,0.00,270.00,0.29,846.02,155.16,0.00,30.88,0.05,0.01,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,250.00,365.04,0.00,347.62,67.56,4687.18,1384.18,889.33,434.40,186.17,99.57,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,3420.00,1208.01,4487.69,427.66,300.29,11163.78,3439.52,4520.82,1273.90,624.63,376.26,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,8550.00,2961.61,15558.20,2106.86,1492.12,28528.64,8624.29,13341.95,3497.26,1904.24,1357.27,NaN,NaN,NaN


## Pharmacy

In [37]:
pharmacy.head()

,pharmacy_order_id,encounter_id,patient_id,ndc_code,drug_name,therapeutic_class,dosage_form,dose_strength,administration_frequency,days_supply,quantity_dispensed,unit_cost_usd,total_drug_cost,insurance_covered_drug_cost,patient_copay_drug_cost,high_risk_medication_flag
0,RX-100001,ENC-100001,PAT-10001,87500-521-28,Ceftriaxone 1g IV,Cephalosporin Antibiotic,IV Infusion,1 g,QD,4,4,20.89,83.56,68.52,15.04,0
1,RX-100002,ENC-100001,PAT-10001,91030-751-41,Nitrofurantoin 100mg,Antibacterial,NaN,100 mg,BID,4,8,1.11,8.88,7.28,1.60,0
2,RX-100003,ENC-100001,PAT-10001,30420-438-16,Phenazopyridine 100mg,Urinary Analgesic,Oral Tablet,100 mg,TID,4,12,0.69,8.28,6.79,1.49,0
3,RX-100004,ENC-100002,PAT-10001,15854-373-53,Azithromycin 500mg,Antibiotic,Oral Tablet,500 mg,QD,30,30,1.88,56.40,46.25,10.15,0
4,RX-100005,ENC-100003,PAT-10001,68208-357-72,Spironolactone 25mg,Aldosterone Antagonist,Oral Tablet,25 mg,QD,30,30,0.48,14.40,11.81,2.59,0


In [38]:
pharmacy.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 26715 entries, 0 to 26714
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   pharmacy_order_id            26715 non-null  str    
 1   encounter_id                 26715 non-null  str    
 2   patient_id                   26715 non-null  str    
 3   ndc_code                     26715 non-null  str    
 4   drug_name                    26715 non-null  str    
 5   therapeutic_class            26715 non-null  str    
 6   dosage_form                  25338 non-null  str    
 7   dose_strength                26715 non-null  str    
 8   administration_frequency     26715 non-null  str    
 9   days_supply                  26715 non-null  int64  
 10  quantity_dispensed           26715 non-null  int64  
 11  unit_cost_usd                26715 non-null  float64
 12  total_drug_cost              26715 non-null  float64
 13  insurance_covered_drug_cost

In [39]:
pharmacy.describe(include='all')

,pharmacy_order_id,encounter_id,patient_id,ndc_code,drug_name,therapeutic_class,dosage_form,dose_strength,administration_frequency,days_supply,quantity_dispensed,unit_cost_usd,total_drug_cost,insurance_covered_drug_cost,patient_copay_drug_cost,high_risk_medication_flag
count,26715,26715,26715,26715,26715,26715,25338,26715,26715,26715.00,26715.00,26715.00,26715.00,26715.00,26715.00,26715.00
unique,26715,13214,5000,26715,41,31,7,30,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,RX-100001,ENC-100001,PAT-11926,87500-521-28,Ceftriaxone 1g IV,Cephalosporin Antibiotic,Oral Tablet,1 g,QD,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,3,17,1,2115,2115,11780,2926,11202,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.16,44.37,16.57,686.49,529.05,157.44,0.20
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.82,54.56,34.71,1741.92,1406.55,558.42,0.40
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00,1.00,0.18,0.28,0.00,0.03,0.00
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.00,10.00,0.80,13.80,8.36,2.49,0.00
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.00,30.00,5.09,97.20,65.68,17.39,0.00
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.00,60.00,20.77,605.01,440.04,108.00,0.00


## Relaciones entre datasets y decisión de unión

El dataset principal que usaremos es donde se encuentra la variable in_hospital_mortality que es la que nos va a servir para predecir de ahí para adelante se usará uniones con las otras tablas con la siguiente llave:

- `visit_encounters.csv`: Principal.
- `clinical_details.csv`: encounter_id.
- `hospital_billing.csv`: encounter_id.
- `patients.csv`: patient_id.
- `pharmacy_prescriptions.csv`: encounter_id.

Adicionalmente en la tabla de pharmacy al tener la posibilidad de tener varias prescripciones medicas para una misma hospitalización lo que se aplicará son funciones agregadas

In [40]:
pharm_agg = (
    pharmacy
    .groupby("encounter_id")
    .agg(
        n_meds               = ("pharmacy_order_id", "count"),
        n_high_risk_meds     = ("high_risk_medication_flag", "sum"),
        total_drug_cost      = ("total_drug_cost", "sum"),
        patient_copay_drugs  = ("patient_copay_drug_cost", "sum"),
      )
    .reset_index()
)


df = (
    encounters
    .merge(clinical,  on=["encounter_id", "patient_id"], how="left", suffixes=("", "_clin"))
    .merge(billing,  on=["encounter_id", "patient_id"], how="left", suffixes=("", "_bill"))
    .merge(patients,   on="patient_id", how="left", suffixes=("", "_pat"))
    .merge(pharm_agg, on="encounter_id", how="left")
)

print(df.shape)         


(13214, 78)


In [41]:
df.head()

,encounter_id,patient_id,encounter_type,admission_datetime,discharge_datetime,length_of_stay_days,admission_urgency,department,attending_physician_id,icu_admitted,icu_days,discharge_disposition,news2_acuity_score,revisit_count,previous_encounter_id,days_since_last_discharge,readmission_30d,in_hospital_mortality,clinical_record_id,primary_diagnosis_code,...,age,blood_group,marital_status,race_ethnicity,residential_zone,socioeconomic_tier,primary_insurance_provider,insurance_policy_id,contact_phone,emergency_contact_phone,street_address,zip_code,baseline_bmi,smoker_status,baseline_charlson_comorbidity_index,registration_date,n_meds,n_high_risk_meds,total_drug_cost,patient_copay_drugs
0,ENC-100001,PAT-10001,Emergency,2023-03-30 00:00:00,2023-04-03 07:12:00,4.30,Emergency,General Medicine,DOC-106,0,0.00,Skilled Nursing Facility,1,1,NaN,NaN,0,0,CLN-100001,N39.0,...,77,O+,Married,Hispanic,Suburban,High,Medicare,INS-887201,533-521-8196x001,790.683.8637x94026,51161 Abbott Rapids,35172,35.70,Former,1,2021-10-02,3,0,100.72,18.13
1,ENC-100002,PAT-10001,Outpatient,2023-04-06 12:00:00,2023-04-06 16:19:12,0.18,Elective,General Medicine,DOC-151,0,0.00,Home,2,2,ENC-100001,3.20,0,0,CLN-100002,J44.1,...,77,O+,Married,Hispanic,Suburban,High,Medicare,INS-887201,533-521-8196x001,790.683.8637x94026,51161 Abbott Rapids,35172,35.70,Former,1,2021-10-02,1,0,56.40,10.15
2,ENC-100003,PAT-10001,Outpatient,2023-04-18 04:19:12,2023-04-18 13:55:12,0.40,Elective,Cardiology,DOC-111,0,0.00,Home,9,3,ENC-100002,11.50,1,0,CLN-100003,I50.9,...,77,O+,Married,Hispanic,Suburban,High,Medicare,INS-887201,533-521-8196x001,790.683.8637x94026,51161 Abbott Rapids,35172,35.70,Former,1,2021-10-02,1,0,14.40,2.59
3,ENC-100004,PAT-10002,Emergency,2023-03-28 00:00:00,2023-04-07 19:12:00,10.80,Emergency,Cardiology,DOC-115,0,0.00,Home,7,1,NaN,NaN,1,0,CLN-100004,I21.9,...,40,B-,Married,Caucasian,Suburban,Middle,Self-Pay,NaN,+1-516-318-4959,NaN,3419 Amanda Gardens Apt. 764,84387,30.00,Never,3,2021-08-24,1,0,4.73,4.73
4,ENC-100005,PAT-10002,Inpatient,2023-04-09 12:00:00,2023-04-12 16:48:00,3.20,Urgent,General Medicine,DOC-134,0,0.00,Home,0,2,ENC-100004,1.70,1,0,CLN-100005,J18.9,...,40,B-,Married,Caucasian,Suburban,Middle,Self-Pay,NaN,+1-516-318-4959,NaN,3419 Amanda Gardens Apt. 764,84387,30.00,Never,3,2021-08-24,3,0,89.55,89.55


In [42]:
#Exportar el dataframe final a un archivo CSV
df.to_csv("../data/merged_df.csv", index=False)